# AI Research Foundations Multilingual Tokenization Challenge


## Imports

In [21]:
from collections import Counter
from importlib.metadata import version
from pathlib import Path
import statistics
import sys
import time
import unicodedata

import pandas as pd
import regex
import tokenizers
from datasets import load_dataset
from tqdm.auto import tqdm
from tokenizers import Tokenizer, decoders, models, normalizers
from tokenizers.models import Unigram
from tokenizers.normalizers import NFC, Prepend
from tokenizers.normalizers import Sequence as NormalizerSequence
from tokenizers.pre_tokenizers import ByteLevel, Metaspace, PreTokenizer, Sequence
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.decoders import Replace as ReplaceDecoder
from tokenizers.decoders import Sequence as DecoderSequence
from tokenizers.decoders import Strip as StripDecoder
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import UnigramTrainer

TOKENIZERS_VERSION = "0.22.1"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
MAX_VOCAB_SIZE = 10_000
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>"]
METASPACE_CHAR = "▁"
SUBMISSION_PATH = "tokenizer.json"

if version("tokenizers") != TOKENIZERS_VERSION:
    raise RuntimeError(
        f"This challenge requires tokenizers=={TOKENIZERS_VERSION}; "
        f"found {tokenizers.__version__}."
    )

print("tokenizers version:", tokenizers.__version__)
print("Maximum vocabulary:", MAX_VOCAB_SIZE)


tokenizers version: 0.22.1
Maximum vocabulary: 10000


## Load the  dataset

In [22]:
def load_competition_data(split):
    """Load the competition data and split as a DataFrame.

    Args:
        split: Either `train` for tokenizer fitting or `validation` for
            evaluation.

    Returns:
        A DataFrame with normalized `language` and NFC-normalized `text`
        columns.
    """
    if split not in {"train", "validation"}:
        raise ValueError("split must be 'train' or 'validation'")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION)
    frame = dataset.to_pandas()[["language", "text"]].copy()
    frame["language"] = frame["language"].str.lower().str.strip()
    frame["text"] = frame["text"].map(lambda text: unicodedata.normalize("NFC", text))
    return frame.reset_index(drop=True)


train = load_competition_data("train")
validation = load_competition_data("validation")
assert set(train.language) == set(validation.language) == set(LANGUAGES), "missing languages"

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
display(train.groupby("language").agg(
    rows=("text", "size"),
    characters=("text", lambda texts: texts.str.len().sum()),
))


Train rows: 240,000
Validation rows: 24,000


,rows,characters
language,,
am,40000,4079673
en,40000,5429908
fr,40000,5686404
ha,40000,5251977
sw,40000,4376925
yo,40000,4629899


## Grapheme-safe training chunks

Training splits only at whitespace boundaries after complete Unicode grapheme clusters. This avoids splitting a base character from its combining marks.


In [23]:
GRAPHEME_BOUNDARY = regex.compile(r"(?<=\X)(?<=\S)(?=\s)")


class GraphemeClusterChunker:
    """Split text at safe whitespace boundaries between grapheme clusters."""

    def split(self, index, normalized_string):
        """Return grapheme-safe slices for the tokenizer pre-tokenizer.

        Args:
            index: Index supplied by the `tokenizers` pre-tokenizer API.
            normalized_string: Text object supplied by that API.

        Returns:
            A list of slices that do not divide a Unicode grapheme cluster.
        """
        text = str(normalized_string)
        if not text:
            return [normalized_string]
        boundaries = [0, *(
            match.start() for match in GRAPHEME_BOUNDARY.finditer(text)
        ), len(text)]
        return [
            normalized_string[start:end]
            for start, end in zip(boundaries[:-1], boundaries[1:])
        ]

    def pre_tokenize(self, pretok):
        """Register grapheme-safe splitting with the tokenizer API.

        Args:
            pretok: `tokenizers` pre-tokenizer object to split in place.
        """
        pretok.split(self.split)


chunker = GraphemeClusterChunker()
metaspace = Metaspace(replacement=METASPACE_CHAR, prepend_scheme="never")
chunk_pretokenizer = Sequence([PreTokenizer.custom(chunker), metaspace])


In [24]:
BYTE_ALPHABET = ByteLevel.alphabet()
assert len(BYTE_ALPHABET) == 256

tokenizer = Tokenizer(Unigram())
text_normalizer = NormalizerSequence([NFC(), Prepend(" ")])
trainer = UnigramTrainer(
    vocab_size=MAX_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=BYTE_ALPHABET,
    show_progress=True,
)


LANGUAGE_REPEAT = {language: 1 for language in LANGUAGES}
LANGUAGE_REPEAT["am"] = 2

weighted_training_texts = []
for language in LANGUAGES:
    texts = train.loc[train.language == language, "text"].tolist()
    weighted_training_texts.extend(texts * LANGUAGE_REPEAT[language])

print("Language repeat factors:", LANGUAGE_REPEAT)
print("Weighted training sentences:", f"{len(weighted_training_texts):,}")


Language repeat factors: {'en': 1, 'fr': 1, 'ha': 1, 'sw': 1, 'yo': 1, 'am': 2}
Weighted training sentences: 280,000


In [25]:
def iter_training_chunks(texts):
    """Yield NFC-normalized, grapheme-safe chunks for Unigram training.

    Args:
        texts: Weighted iterable of raw training strings Yields:

        
        Non-empty metaspace-processed chunks for tokenizer vocabulary learning.
    """
    total_chunks = 0
    progress = tqdm(
        texts,
        total=len(texts),
        desc="Grapheme-safe pre-tokenizing",
        unit="sentences",
        dynamic_ncols=True,
    )
    for text in progress:
        normalized = text_normalizer.normalize_str(text)
        if not normalized:
            continue
        for chunk, _ in chunk_pretokenizer.pre_tokenize_str(normalized):
            if chunk:
                total_chunks += 1
                yield chunk
        progress.set_postfix(chunks=f"{total_chunks:,}", refresh=False)
    print(f"Training chunks streamed: {total_chunks:,}")


In [26]:
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=False)

started = time.perf_counter()
tokenizer.train_from_iterator(
    iter_training_chunks(weighted_training_texts),
    trainer=trainer,
    length=len(weighted_training_texts),
)
training_seconds = time.perf_counter() - started

print(f"Training time: {training_seconds:.2f} seconds")
print(f"Vocabulary size: {tokenizer.get_vocab_size(with_added_tokens=True):,}")


Grapheme-safe pre-tokenizing:   0%|                                                                           …

Training chunks streamed: 5,822,527


Training time: 250.76 seconds
Vocabulary size: 10,000


In [27]:
tokenizer.normalizer = text_normalizer
tokenizer.pre_tokenizer = Sequence([
    metaspace,
    ByteLevel(add_prefix_space=False, use_regex=False),
])
tokenizer.decoder = DecoderSequence([
    ByteLevelDecoder(add_prefix_space=False),
    StripDecoder(METASPACE_CHAR, 1, 0),
    ReplaceDecoder(METASPACE_CHAR, " "),
])

bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <eos> <bos> $B <eos>",
    special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
)

vocab_size = tokenizer.get_vocab_size(with_added_tokens=True)
assert vocab_size <= MAX_VOCAB_SIZE, (vocab_size, MAX_VOCAB_SIZE)
assert tokenizer.token_to_id("<unk>") is None

tokenizer.save(SUBMISSION_PATH, pretty=True)
print(f"Saved {SUBMISSION_PATH} with {vocab_size:,} / {MAX_VOCAB_SIZE:,} tokens")


Saved tokenizer.json with 10,000 / 10,000 tokens


## Official validation score and submission checks
 It reports fertility, unknown-token rate, guardrail and reconstruction penalties, file validity, and encoding throughput.

In [28]:
def ensure_utils():
    """Add the supplied local competition checker to Python's import path.

    Raises:
        FileNotFoundError: If neither supported local `utils.py` location is
            available.
    """
    for root in (Path.cwd(), *Path.cwd().parents):
        for relative in ("utils.py", "starter/utils.py"):
            candidate = root / relative
            if candidate.is_file():
                location = str(candidate.parent)
                if location not in sys.path:
                    sys.path.insert(0, location)
                return
    raise FileNotFoundError("Place the supplied competition utils.py beside this notebook.")


ensure_utils()
from utils import profile_submission

validation_report = profile_submission(SUBMISSION_PATH, data=validation, repeats=3)
assert validation_report["valid"], validation_report["errors"]
assert validation_report["lossy_rows"] == 0
print(f"Official validation score: {validation_report['score']:.4f}")


AI Research Foundations Multilingual Tokenization Challenge
Submission checker

Loading tokenizer......... ✓
File size................. ✓
Vocabulary................ ✓ 10,000 / 10,000
Encoding.................. ✓
Compatibility............. ✓

Scores (24,000 rows, lower is better)
language      tokens/word  [UNK] rate    score
English             1.914      0.0000    1.914
French              1.974      0.0000    1.974
Hausa*              1.739      0.0000    1.739
Swahili*            1.976      0.0000    1.976
Yoruba*             1.963      0.0000    1.963
Amharic*            2.450      0.0000    2.450
Guardrail penalty......... 0.0000
Reconstruction penalty.... 0.0000
SCORE..................... 2.0317
  * scored languages
  guardrail 2.336, highest is French at 1.974 (15.5% headroom)
  reconstruction 100.0%, charged 3 x the share not reconstructed

Local benchmark (informational only)
Evaluation time........... 1.31 s
Throughput................ 2.2M characters/sec

READY FOR SUBMISSION

## Character-baseline runtime comparison


In [29]:
def train_character_baseline(texts, min_count=20):
    """Build the byte-backed character baseline used for runtime comparison.

    Args:
        texts: Training strings used to count frequent characters.
        min_count: Minimum character frequency for a dedicated vocabulary entry.

    Returns:
        A character-level BPE tokenizer with byte fallback.
    """
    counts = Counter(character for text in texts for character in text)
    alphabet = [character for character, count in counts.most_common() if count >= min_count]
    vocab = {"[UNK]": 0}
    for character in alphabet:
        vocab.setdefault(character, len(vocab))
    for value in range(256):
        vocab.setdefault(f"<0x{value:02X}>", len(vocab))
    baseline = Tokenizer(models.BPE(
        vocab=vocab, merges=[], unk_token="[UNK]", byte_fallback=True,
    ))
    baseline.normalizer = normalizers.NFC()
    baseline.decoder = decoders.ByteFallback()
    return baseline


def median_encode_seconds(model, texts, repeats=3):
    """Measure median batch-encoding time after one warm-up pass.

    Args:
        model: A trained `tokenizers.Tokenizer`.
        texts: Strings to encode as one batch.
        repeats: Number of timed encoding passes.

    Returns:
        Median elapsed encoding time in seconds.
    """
    model.encode_batch(texts, add_special_tokens=False)
    timings = []
    for _ in range(repeats):
        started = time.perf_counter()
        model.encode_batch(texts, add_special_tokens=False)
        timings.append(time.perf_counter() - started)
    return statistics.median(timings)


validation_texts = validation.text.tolist()
character_baseline = train_character_baseline(train.text.tolist())
baseline_seconds = median_encode_seconds(character_baseline, validation_texts)
candidate_seconds = median_encode_seconds(tokenizer, validation_texts)
slowdown = candidate_seconds / max(baseline_seconds, 1e-12)

print(f"Character baseline: {baseline_seconds:.3f}s")
print(f"Candidate tokenizer: {candidate_seconds:.3f}s")
print(f"Slowdown: {slowdown:.2f}× (limit: 5.00×)")
if slowdown > 5:
    print("WARNING: local benchmark exceeds the competition runtime limit.")


Character baseline: 0.530s
Candidate tokenizer: 1.330s
Slowdown: 2.51× (limit: 5.00×)


## Submission files

- `tokenizer.json`


In [30]:
assert Path(SUBMISSION_PATH).is_file()
assert tokenizer.get_vocab_size(with_added_tokens=True) <= MAX_VOCAB_SIZE
assert validation_report["valid"]

print("Final local score:", f"{validation_report['score']:.4f}")


Final local score: 2.0317
